In [1]:
import pandas as pd

df = pd.read_csv("../data/Womens Clothing E-Commerce Reviews.csv")

print(df.shape)

(23486, 11)


In [2]:
low_rated = df[df["Rating"] <= 3].copy()

print("Total low-rated reviews:", len(low_rated))

Total low-rated reviews: 5278


In [3]:
sample_200 = low_rated.sample(n=200, random_state=42).copy()

print("Sample size:", len(sample_200))
sample_200.head()

Sample size: 200


,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
26,26,949,33,Huge disappointment,I have been waiting for this sweater coat to s...,2,0,0,General,Tops,Sweaters
18807,18807,1094,46,Abroad dress,This is a heavy dress - it's surprising when y...,3,0,2,General Petite,Dresses,Dresses
2334,2334,1047,37,Ripped on first day,Perfect summer pants until i wanted to pet my ...,3,0,1,General Petite,Bottoms,Pants
8059,8059,1078,46,Wrinkled!,The dress arrived wrinkled so i washed it and ...,1,0,0,General Petite,Dresses,Dresses
17498,17498,1081,24,"Almost perfect, but the top is unflattering.","The dress had great potential, but the top sag...",3,0,0,General Petite,Dresses,Dresses


In [4]:
print("Missing review text:", sample_200["Review Text"].isna().sum())
print("Rating distribution:")
print(sample_200["Rating"].value_counts().sort_index())

Missing review text: 7
Rating distribution:
Rating
1     27
2     55
3    118
Name: count, dtype: int64


In [5]:
sample_200["Review Text"] = sample_200["Review Text"].fillna("")

llm_reviews = sample_200[sample_200["Review Text"].str.strip() != ""].copy()

print("Reviews with text for LLM:", len(llm_reviews))
print("Reviews excluded due to missing text:", len(sample_200) - len(llm_reviews))

Reviews with text for LLM: 193
Reviews excluded due to missing text: 7


In [6]:
labels = [
    "size/fit",
    "fabric/quality",
    "looks different from photo",
    "comfort",
    "price",
    "other"
]

prompt_template = """
You are classifying customer reviews for a women's clothing retailer.

Choose exactly ONE main issue from these labels:
1. size/fit
2. fabric/quality
3. looks different from photo
4. comfort
5. price
6. other

Also identify whether the review says the item runs:
- small
- large
- neither

Return ONLY this format:
issue|fit

Do not add explanations.

Review:
{review}
"""

print("Labels:", labels)
print(prompt_template)

Labels: ['size/fit', 'fabric/quality', 'looks different from photo', 'comfort', 'price', 'other']

You are classifying customer reviews for a women's clothing retailer.

Choose exactly ONE main issue from these labels:
1. size/fit
2. fabric/quality
3. looks different from photo
4. comfort
5. price
6. other

Also identify whether the review says the item runs:
- small
- large
- neither

Return ONLY this format:
issue|fit

Do not add explanations.

Review:
{review}



In [7]:
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

print("Groq client connected successfully!")

Groq client connected successfully!


In [9]:
test_review = llm_reviews.iloc[0]["Review Text"]

test_prompt = prompt_template.format(review=test_review)

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": test_prompt
        }
    ],
    temperature=0
)

print("Review:")
print(test_review)

print("\nLLM Output:")
print(response.choices[0].message.content)

Review:
I have been waiting for this sweater coat to ship for weeks and i was so excited for it to arrive. this coat is not true to size and made me look short and squat. the sleeves are very wide (although long). as a light weight fall coat the sleeves don't need to be as wide because you wouldn't be layerng too much underneath. the buttons need to be moved at least three inches in for a nicer fit. i thought about redoing the buttons myself but the sleeves looked even more out of proportion with a tigh

LLM Output:
size/fit|small


In [10]:
llm_output = response.choices[0].message.content.strip()

parts = llm_output.split("|")

print("Raw output:", llm_output)
print("Number of parts:", len(parts))

if len(parts) == 2:
    print("Issue:", parts[0].strip())
    print("Fit:", parts[1].strip())

Raw output: size/fit|small
Number of parts: 2
Issue: size/fit
Fit: small


In [11]:
import time

results = []

for _, row in llm_reviews.iterrows():
    review = row["Review Text"]
    prompt = prompt_template.format(review=review)

    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        raw_output = response.choices[0].message.content.strip()

        parts = raw_output.split("|")

        if len(parts) == 2:
            issue = parts[0].strip()
            fit = parts[1].strip()
        else:
            issue = "parse_error"
            fit = "parse_error"

    except Exception as e:
        raw_output = str(e)
        issue = "error"
        fit = "error"

    results.append({
        "Review ID": row["Unnamed: 0"],
        "Clothing ID": row["Clothing ID"],
        "Department Name": row["Department Name"],
        "Rating": row["Rating"],
        "Review Text": review,
        "Issue": issue,
        "Fit": fit,
        "Raw Output": raw_output
    })

    time.sleep(0.1)

llm_results = pd.DataFrame(results)

print("Reviews processed:", len(llm_results))
print("\nIssue counts:")
print(llm_results["Issue"].value_counts())

print("\nFit counts:")
print(llm_results["Fit"].value_counts())

Reviews processed: 193

Issue counts:
Issue
size/fit                      137
fabric/quality                 29
looks different from photo     12
other                           7
parse_error                     3
price                           2
comfort                         2
2                               1
Name: count, dtype: int64

Fit counts:
Fit
neither        72
large          64
small          54
parse_error     3
Name: count, dtype: int64


In [12]:
invalid_results = llm_results[
    ~llm_results["Issue"].isin(labels)
]

print(invalid_results[
    ["Review ID", "Review Text", "Issue", "Fit", "Raw Output"]
].to_string(index=False))

 Review ID                                                                                                                                                                                                                                                                                                                                                                                                                Review Text       Issue         Fit                                                            Raw Output
     19115                                                                                                            Love the way this dress looks online. sadly, it doesn't look great on me. i'm 5'4", athletic build. the small fit perfectly, but the extra fabric in the hip/pocket area just seemed to drag me down. i really wished this worked for me, but i think you've gotta be a little bit taller to pull this one off. parse_error parse_error                                       

In [13]:
invalid_indices = llm_results[
    ~llm_results["Issue"].isin(labels)
].index

for idx in invalid_indices:
    review = llm_results.loc[idx, "Review Text"]
    prompt = prompt_template.format(review=review)

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    raw_output = response.choices[0].message.content.strip()

    print("=" * 80)
    print("Review ID:", llm_results.loc[idx, "Review ID"])
    print("New LLM Output:", raw_output)

Review ID: 19115
New LLM Output: size/fit|neither
Review ID: 22126
New LLM Output: 2|neither
Review ID: 22824
New LLM Output: looks different from photo|neither
size/fit|neither
comfort|neither
Review ID: 10197
New LLM Output: 


In [14]:
print(
    llm_results.loc[
        llm_results["Review ID"] == 10197,
        ["Review ID", "Review Text", "Raw Output"]
    ].to_string(index=False)
)

 Review ID                                         Review Text Raw Output
     10197 Large and the material is too bulky. i returned it.           


In [15]:
manual_30 = llm_results.sample(n=30, random_state=123).copy()

manual_30[
    ["Review ID", "Rating", "Department Name", "Review Text", "Issue", "Fit"]
]

,Review ID,Rating,Department Name,Review Text,Issue,Fit
72,16442,3,Bottoms,Super cute skirt. i decided to return it becau...,other,neither
4,17498,3,Dresses,"The dress had great potential, but the top sag...",size/fit,neither
79,1077,3,Tops,This top was quite voluminous when i purchased...,size/fit,small
171,3695,3,Tops,Unfortunately this piece has a very boxy cut a...,size/fit,neither
112,11415,3,Bottoms,"Nice soft jeans, cute style, and i wish i coul...",size/fit,small
71,11876,1,Bottoms,I tried on this jumpsuit in black and it was a...,size/fit,small
90,17204,2,Dresses,I bought a petite medium and the length was ri...,size/fit,large
93,5683,3,Dresses,I was excited to receive this dress because it...,size/fit,small
182,1196,3,Dresses,"I am 5'-7"" and 135 lbs, i bought a medium peti...",size/fit,large
147,609,3,Dresses,"I just tried on this dress in the store, in re...",size/fit,neither


In [16]:
manual_check = manual_30[
    ["Review ID", "Rating", "Department Name", "Review Text", "Issue", "Fit"]
].copy()

manual_check["Manual_Issue"] = ""
manual_check["Manual_Fit"] = ""

manual_check

,Review ID,Rating,Department Name,Review Text,Issue,Fit,Manual_Issue,Manual_Fit
72,16442,3,Bottoms,Super cute skirt. i decided to return it becau...,other,neither,,
4,17498,3,Dresses,"The dress had great potential, but the top sag...",size/fit,neither,,
79,1077,3,Tops,This top was quite voluminous when i purchased...,size/fit,small,,
171,3695,3,Tops,Unfortunately this piece has a very boxy cut a...,size/fit,neither,,
112,11415,3,Bottoms,"Nice soft jeans, cute style, and i wish i coul...",size/fit,small,,
71,11876,1,Bottoms,I tried on this jumpsuit in black and it was a...,size/fit,small,,
90,17204,2,Dresses,I bought a petite medium and the length was ri...,size/fit,large,,
93,5683,3,Dresses,I was excited to receive this dress because it...,size/fit,small,,
182,1196,3,Dresses,"I am 5'-7"" and 135 lbs, i bought a medium peti...",size/fit,large,,
147,609,3,Dresses,"I just tried on this dress in the store, in re...",size/fit,neither,,


In [17]:
row = manual_check.iloc[0]

print("Review ID:", row["Review ID"])
print("Rating:", row["Rating"])
print("Department:", row["Department Name"])

print("\nREVIEW:")
print(row["Review Text"])

print("\nLLM prediction:")
print("Issue:", row["Issue"])
print("Fit:", row["Fit"])

Review ID: 16442
Rating: 3
Department: Bottoms

REVIEW:
Super cute skirt. i decided to return it because i felt like i did't have enough tops to go with it.

LLM prediction:
Issue: other
Fit: neither


In [18]:
row = manual_check.iloc[1]

print("Review ID:", row["Review ID"])
print("Rating:", row["Rating"])
print("Department:", row["Department Name"])

print("\nREVIEW:")
print(row["Review Text"])

print("\nLLM prediction:")
print("Issue:", row["Issue"])
print("Fit:", row["Fit"])

Review ID: 17498
Rating: 3
Department: Dresses

REVIEW:
The dress had great potential, but the top sagged and made the dress look unflattering on me. unfortunately, i had to return it. the length was great, and the back and sides of the dress were nicely tailored and flattering. perhaps this would fit someone else better - i have a 34b/34c bust, and it didn't work for me.

LLM prediction:
Issue: size/fit
Fit: neither


In [19]:
for i, (_, row) in enumerate(manual_check.iterrows(), start=1):
    print("=" * 100)
    print(f"REVIEW {i}/30")
    print("Review ID:", row["Review ID"])
    print("Rating:", row["Rating"])
    print("Department:", row["Department Name"])
    print("\nREVIEW:")
    print(row["Review Text"])
    print("\nLLM prediction:")
    print("Issue:", row["Issue"])
    print("Fit:", row["Fit"])
    print()

REVIEW 1/30
Review ID: 16442
Rating: 3
Department: Bottoms

REVIEW:
Super cute skirt. i decided to return it because i felt like i did't have enough tops to go with it.

LLM prediction:
Issue: other
Fit: neither

REVIEW 2/30
Review ID: 17498
Rating: 3
Department: Dresses

REVIEW:
The dress had great potential, but the top sagged and made the dress look unflattering on me. unfortunately, i had to return it. the length was great, and the back and sides of the dress were nicely tailored and flattering. perhaps this would fit someone else better - i have a 34b/34c bust, and it didn't work for me.

LLM prediction:
Issue: size/fit
Fit: neither

REVIEW 3/30
Review ID: 1077
Rating: 3
Department: Tops

REVIEW:
This top was quite voluminous when i purchased it and i sized down to an xs, however after hand washing it once in cold water it has shrunk significantly. it still fits, but something to be aware of.

LLM prediction:
Issue: size/fit
Fit: small

REVIEW 4/30
Review ID: 3695
Rating: 3
Depart

In [20]:
pd.set_option("display.max_colwidth", None)

manual_check[
    ["Review ID", "Rating", "Department Name", "Review Text", "Issue", "Fit"]
].to_string(index=False)

' Review ID  Rating Department Name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              Review Text                      Issue         Fit\n     16442       3         Bottoms                                                                                                                                                                                                                                                                                                                                                                                                 

In [21]:
manual_check[
    ["Review ID", "Rating", "Department Name", "Review Text", "Issue", "Fit"]
].to_csv("../outputs/manual_30_check.csv", index=False)

print("Saved: outputs/manual_30_check.csv")

Saved: outputs/manual_30_check.csv


In [22]:
df[df["Unnamed: 0"] == 20477][["Unnamed: 0", "Review Text"]]

,Unnamed: 0,Review Text
20477,20477,"Okay, so this top has its redeeming qualities, i promise! which is why i gave it 3 stars when i really wanted to give it two. but it's just not worth 2 because i'm sure it works for some people and the quality is just impeccable. for people who purchase this, it will last forever. it's sturdy cotton, not thin like many retailer tee materials. the lace detail and texture is on point and the neckline is very nice. it's a great looking top. my issue though, wasn't with the bust as other reviewers hav"


In [23]:
print(df.loc[df["Unnamed: 0"] == 20477, "Review Text"].iloc[0])

Okay, so this top has its redeeming qualities, i promise! which is why i gave it 3 stars when i really wanted to give it two. but it's just not worth 2 because i'm sure it works for some people and the quality is just impeccable. for people who purchase this, it will last forever. it's sturdy cotton, not thin like many retailer tee materials. the lace detail and texture is on point and the neckline is very nice. it's a great looking top. my issue though, wasn't with the bust as other reviewers hav


In [24]:
manual_labels = {
    16442: ("other", "neither"),
    17498: ("size/fit", "neither"),
    1077: ("fabric/quality", "small"),
    3695: ("size/fit", "neither"),
    11415: ("size/fit", "small"),
    11876: ("fabric/quality", "neither"),
    17204: ("size/fit", "large"),
    5683: ("size/fit", "small"),
    1196: ("size/fit", "large"),
    609: ("size/fit", "neither"),
    16227: ("size/fit", "small"),
    4469: ("fabric/quality", "neither"),
    1273: ("size/fit", "small"),
    13572: ("size/fit", "neither"),
    12900: ("size/fit", "large"),
    19850: ("looks different from photo", "neither"),
    7215: ("size/fit", "neither"),
    9338: ("size/fit", "large"),
    21305: ("fabric/quality", "large"),
    19248: ("size/fit", "large"),
    22824: ("looks different from photo", "neither"),
    7847: ("size/fit", "small"),
    5564: ("looks different from photo", "neither"),
    22576: ("size/fit", "small"),
    3557: ("size/fit", "small"),
    7184: ("size/fit", "neither"),
    20477: ("other", "neither"),
    2554: ("comfort", "large"),
    4033: ("size/fit", "neither"),
    12894: ("fabric/quality", "neither")
}

manual_check["Manual_Issue"] = manual_check["Review ID"].map(
    lambda x: manual_labels[x][0]
)

manual_check["Manual_Fit"] = manual_check["Review ID"].map(
    lambda x: manual_labels[x][1]
)

manual_check["Issue_Match"] = (
    manual_check["Issue"] == manual_check["Manual_Issue"]
)

manual_check["Fit_Match"] = (
    manual_check["Fit"] == manual_check["Manual_Fit"]
)

print("Issue accuracy:",
      manual_check["Issue_Match"].mean() * 100, "%")

print("Fit accuracy:",
      manual_check["Fit_Match"].mean() * 100, "%")

print("Overall exact-match accuracy:",
      (
          manual_check["Issue_Match"] &
          manual_check["Fit_Match"]
      ).mean() * 100, "%")

Issue accuracy: 83.33333333333334 %
Fit accuracy: 86.66666666666667 %
Overall exact-match accuracy: 80.0 %


In [25]:
issue_by_department = (
    llm_results
    .groupby(["Department Name", "Issue"])
    .size()
    .unstack(fill_value=0)
)

print(issue_by_department)

Issue            2  comfort  fabric/quality  looks different from photo  \
Department Name                                                           
Bottoms          0        0               4                           1   
Dresses          0        0              10                           4   
Intimate         1        0               0                           1   
Jackets          0        0               2                           0   
Tops             0        2              13                           6   
Trend            0        0               0                           0   

Issue            other  parse_error  price  size/fit  
Department Name                                       
Bottoms              1            1      0        21  
Dresses              2            1      0        48  
Intimate             1            0      0         9  
Jackets              0            0      1         4  
Tops                 3            1      0        54  
Trend         

In [26]:
llm_results["Final_Issue"] = llm_results["Issue"]

fixes = {
    19115: "size/fit",
    22126: "comfort",
    22824: "looks different from photo",
    10197: "fabric/quality"
}

for review_id, issue in fixes.items():
    llm_results.loc[
        llm_results["Review ID"] == review_id,
        "Final_Issue"
    ] = issue

issue_by_department = (
    llm_results
    .groupby(["Department Name", "Final_Issue"])
    .size()
    .unstack(fill_value=0)
)

print(issue_by_department)

Final_Issue      comfort  fabric/quality  looks different from photo  other  \
Department Name                                                               
Bottoms                0               4                           2      1   
Dresses                0              10                           4      2   
Intimate               1               0                           1      1   
Jackets                0               2                           0      0   
Tops                   2              14                           6      3   
Trend                  0               0                           0      0   

Final_Issue      price  size/fit  
Department Name                   
Bottoms              0        21  
Dresses              0        49  
Intimate             0         9  
Jackets              1         4  
Tops                 0        54  
Trend                1         1  


In [27]:
llm_results.to_csv(
    "../outputs/llm_tagged_193_reviews.csv",
    index=False
)

manual_check.to_csv(
    "../outputs/manual_30_check.csv",
    index=False
)

print("Saved: outputs/llm_tagged_193_reviews.csv")
print("Saved: outputs/manual_30_check.csv")

Saved: outputs/llm_tagged_193_reviews.csv
Saved: outputs/manual_30_check.csv
